# Hex6 Fast Bootstrap on Colab

This notebook runs the reduced `configs/fast.toml` training profile.
Use it to validate the training loop quickly before scaling up search or model size.

In [ ]:
REPO_MODE = "git"  # set to "drive" to use a repo folder in Google Drive
REPO_URL = "https://github.com/Stroudmj00/hex6-bot.git"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Hexagonal tic tac toe"
WORKDIR = "/content/hex6-bot"
ARTIFACT_TARGET = "/content/drive/MyDrive/hex6_artifacts/bootstrap_fast"


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    shutil.copytree(DRIVE_REPO_PATH, WORKDIR)
else:
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("WORKDIR:", os.getcwd())


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "numpy"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run(
    [
        "python",
        "-c",
        "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')",
    ],
    check=True,
)


In [ ]:
subprocess.run(
    [
        "python",
        "-m",
        "hex6.train.run_bootstrap",
        "--config",
        "configs/fast.toml",
        "--output",
        "artifacts/bootstrap_fast",
    ],
    check=True,
)


In [ ]:
metrics_path = Path("artifacts/bootstrap_fast/metrics.json")
print(metrics_path.read_text())

if REPO_MODE == "drive":
    target = Path(ARTIFACT_TARGET)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree("artifacts/bootstrap_fast", target)
    print("Artifacts copied to", target)
